# Learning with constraints: forward-backward, Baum–Welch, and GEM

There are two flavors of learning that maximize different objectives with a constraint set $C$:

1. The joint likelihood $p_\theta(y,C)$: constraints are additional informaiton
2. The conditional likelihood $p_\theta(y \mid C)$: constraints are conditioning events.


`baum_welch_mvr_chmm` maximizes the former joint likelihood, and is just regular Buam-Welch over the augmented chain (with moment marginalization)
.
`generalized_em_mvr_chmm` is a generalized EM procedure for the latter conditional likelihood. Due to the extra factor $p_\theta(C)$, we lose out on close-form M-steps for the intial vector and transition matrix (emission updates are still closed form). Nonetheless, this procedure enjoys the usual monotone ascent property, and the M-step consists of small gradient steps.

In [ ]:
import itertools
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.learning.baum_welch_mvr import (
    baum_welch_mvr_chmm,
    forward_backward_mvr_chmm,
)

## 1. The model

The same three-state HMM as the inference notebooks. The constraint is a **subsequence** one: `A` is forbidden over time `[0, 3]`. A global constraint that bans `A` is equivalent to a restriction on the parameter space: we use a subsequence constraint to show how path constraint information can be integrated into learning.

In [ ]:
HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

start_probs = {
    "A": 0.28,
    "B": 0.40,
    "C": 0.32,
}

transition_probs = {
    ("A", "A"): 0.34,
    ("A", "B"): 0.05,
    ("A", "C"): 0.61,
    ("B", "A"): 0.48,
    ("B", "B"): 0.06,
    ("B", "C"): 0.46,
    ("C", "A"): 0.43,
    ("C", "B"): 0.18,
    ("C", "C"): 0.39,
}

emission_probs = {
    ("A", "lo"): 0.20,
    ("A", "mid"): 0.31,
    ("A", "hi"): 0.49,
    ("B", "lo"): 0.54,
    ("B", "mid"): 0.23,
    ("B", "hi"): 0.23,
    ("C", "lo"): 0.09,
    ("C", "mid"): 0.01,
    ("C", "hi"): 0.90,
}

true_hmm = HiddenMarkovModel()
true_hmm.load_model(
    start_probs=start_probs,
    transition_probs=transition_probs,
    emission_probs=emission_probs,
    initialize=True,
)

T = 8
WINDOW = [0, 3]
FORBIDDEN_STATE = "A"
forbidden_idx = true_hmm.hidden_to_internal[FORBIDDEN_STATE]


def forbid_mvr(state, time_range=None, name=None):
    """MVR rejecting any path that visits ``state`` inside its window."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        time_range=time_range,
        name=name,
    )


no_A_early = forbid_mvr(FORBIDDEN_STATE, time_range=WINDOW, name="no_A_early")
print(f"constraint: no {FORBIDDEN_STATE} at t in [{WINDOW[0]}, {WINDOW[1]}], horizon T = {T}")

## 2. Forward-backward

`forward_backward_mvr_chmm` returns the posterior over hidden states given the
observations **and** the fact that the path is feasible. The window shows up
directly: `P(A)` is exactly zero for `t <= 3` and free afterwards.

In [ ]:
rng = np.random.default_rng(3)
repn = true_hmm.repn


def sample_constrained(n):
    """Rejection-sample sequences whose hidden path satisfies the constraint."""
    out, tries = [], 0
    while len(out) < n:
        tries += 1
        hidden = [rng.choice(3, p=repn.start_vec)]
        for _ in range(T - 1):
            hidden.append(rng.choice(3, p=repn.transition_mat[hidden[-1]]))
        if forbidden_idx in hidden[WINDOW[0] : WINDOW[1] + 1]:
            continue
        out.append([
            # emission_mat columns follow the HMM's own observed ordering.
            true_hmm.observed_to_external[rng.choice(3, p=repn.emission_mat[h])]
            for h in hidden
        ])
    return out, tries


sequences, tries = sample_constrained(80)
print(f"{len(sequences)} feasible sequences kept from {tries} draws "
      f"({len(sequences) / tries:.1%} acceptance)")

truth = MVR_CHMM(hidden_markov_model=true_hmm, constraints=[no_A_early])
gamma, xi, loglik = forward_backward_mvr_chmm(truth, sequences[0])

print(f"\nloglik = {loglik:.6f}   gamma {tuple(gamma.shape)}   xi {tuple(xi.shape)}")
print()
print(pd.DataFrame(
    np.round(gamma.numpy(), 4), columns=HIDDEN_STATES,
).rename_axis("t").to_string())

### Checked against enumeration

$3^8 = 6561$ paths, so the posterior can be computed by brute-force.

In [ ]:
def brute_force_posterior(obs):
    """Exact gamma and loglik by enumerating every feasible hidden path."""
    idx_of = hmm_idx = true_hmm.hidden_to_internal
    r = true_hmm.repn
    total, marg = 0.0, np.zeros((T, 3))

    for path in itertools.product(range(3), repeat=T):
        if forbidden_idx in path[WINDOW[0] : WINDOW[1] + 1]:
            continue
        w = r.start_vec[path[0]]
        for t in range(1, T):
            w *= r.transition_mat[path[t - 1]][path[t]]
        for t, o in enumerate(obs):
            w *= r.emission_mat[path[t]][true_hmm.observed_to_internal[o]]
        total += w
        for t in range(T):
            marg[t, path[t]] += w

    return marg / total, math.log(total)


exact_gamma, exact_loglik = brute_force_posterior(sequences[0])

print(f"loglik  algorithm {loglik:.10f}   brute force {exact_loglik:.10f}")
print(f"gamma   max abs diff {np.abs(exact_gamma - gamma.numpy()).max():.3e}")
print(f"\nP({FORBIDDEN_STATE}) at t <= {WINDOW[1]}: {np.round(gamma.numpy()[: WINDOW[1] + 1, forbidden_idx], 12)}")
print(f"P({FORBIDDEN_STATE}) at t >  {WINDOW[1]}: {np.round(gamma.numpy()[WINDOW[1] + 1 :, forbidden_idx], 4)}")

## 3. EM: Constraints as Observations vs Constrained Learning
We demonstrate the distinction between: 

1. learning with constraints as additional information about the unconstrained latent path
2. learning a constrained distribution.

For example, consider a constraint that forbids visits to `A`. This can be interpreted as:

1. We drew a run from the unconstrained chain, and our draw didn't visit `A` by chance. This is given as additional information about the latent path.
2. We drew a run from the constrained chain where visits to `A` were forbidden. This is a constrained distribution.


In this experiment, runs are drawn from the constrained model $P(Y \ \ \big| \ C, \ \theta )$ but are incorporated as observation from the unconstrained model $P(Y, \ C \ \big| \ \theta )$. Our model is misspecified. Nonetheless, since EM is known to be monotone, the log-likelihood must increase every iteration even in this misspecified scenario. 

In [ ]:
def flat_hmm():
    """A deliberately uninformative starting point, with no structural zeros."""
    rows = [[0.5, 0.3, 0.2], [0.2, 0.5, 0.3], [0.3, 0.2, 0.5]]
    model = HiddenMarkovModel()
    model.load_model(
        start_probs={h: 1 / 3 for h in HIDDEN_STATES},
        transition_probs={
            (a, b): rows[i][j]
            for i, a in enumerate(HIDDEN_STATES)
            for j, b in enumerate(HIDDEN_STATES)
        },
        emission_probs={
            (h, o): rows[i][j]
            for i, h in enumerate(HIDDEN_STATES)
            for j, o in enumerate(OBSERVED_STATES)
        },
        initialize=True,
    )
    return model


start_model = MVR_CHMM(hidden_markov_model=flat_hmm(), constraints=[no_A_early])
fitted, history = baum_welch_mvr_chmm(start_model, sequences, max_iter=200, tol=1e-9)

steps = np.diff(history)
print(f"{len(history)} iterations")
print(f"loglik {history[0]:.4f} -> {history[-1]:.4f}")
print(f"monotone: {bool((steps >= -1e-9).all())}   smallest step {steps.min():.2e}")

### About that warning

EM ran out of its iteration budget before the change dropped below `tol=1e-9`, so
it warns:

- `history[i]` is the log-likelihood at the **start** of iteration `i`, before that
  iteration's update. So `history[0]` always scores the model passed in.
- If EM *converges*, no update follows the last entry and `history[-1]` scores the
  returned model exactly.
- If it stops at `max_iter` instead — as here — one further update was applied, so
  the returned model is one step **ahead** of `history[-1]`. You can see this
  below: the fitted model scores slightly better than `history[-1]` reports.

Passing `tol=0` will automatically make EM run for `max_iter` iterations and
suppresses the warning.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(history, color="#2E6DB4", linewidth=2)
ax1.set_xlabel("EM iteration")
ax1.set_ylabel(r"$\log P(y,\ \mathrm{constraints}\ |\ \theta)$")
ax1.set_title("Joint log-likelihood", loc="left", fontsize=11)
ax1.grid(color="0.9", linewidth=0.8)
ax1.set_axisbelow(True)

ax2.semilogy(np.maximum(steps, 1e-16), color="#E08A3C", linewidth=2)
ax2.set_xlabel("EM iteration")
ax2.set_ylabel("increase per iteration")
ax2.set_title("Every step is an increase", loc="left", fontsize=11)
ax2.grid(color="0.9", linewidth=0.8)
ax2.set_axisbelow(True)

fig.tight_layout()
plt.show()

## 4. Misspecified: Joint likelihood versus unconstrained learning

The sequences were drawn from the **constrained model**
$P(y\mid C,\theta_\star)$, where $C$ denotes the constraints. Joint Baum–Welch
instead maximizes $\sum_i\log P(y_i,C\mid\theta)$: it treats feasibility as
additional evidence about an ordinary HMM draw, without accounting for the
selection of feasible paths.

Comparing two fits on the same sequences, from the same initialization:

| Fit | Objective |
|---|---|
| Joint BW | $\sum_i\log P(y_i,C_T\mid\theta)$ |
| Unconstrained BW | $\sum_i\log P(y_i\mid\theta)$ |

**Both are misspecified for this experiment.** Joint BW incorporates feasibility
as evidence; unconstrained BW ignores it. Neither fits the conditional law that
generated the sample. The table below scores both on the **joint likelihood** to
show the effect of incorporating constraint information.

In [ ]:
unconstrained = MVR_CHMM(hidden_markov_model=flat_hmm(), constraints=[])
free_fit, _ = baum_welch_mvr_chmm(unconstrained, sequences, max_iter=200, tol=1e-9)


def constrained_loglik(model):
    """Total log P(y, constraints | theta) over the whole data set."""
    chmm = MVR_CHMM(hidden_markov_model=model, constraints=[no_A_early])
    return sum(forward_backward_mvr_chmm(chmm, s)[2] for s in sequences)


rows = [
    ("starting model", flat_hmm()),
    ("unconstrained fit", free_fit),
    ("joint BW fit", fitted),
    ("true model", true_hmm),
]

print(pd.DataFrame(
    [(name, round(constrained_loglik(m), 3)) for name, m in rows],
    columns=["model", "log P(y, constraints | theta)"],
).to_string(index=False))

print(f"\nhistory[0]      = {history[0]:.3f}")
print(f"starting model  = {constrained_loglik(flat_hmm()):.3f}   (history[0] scores the model passed in)")

In this run, joint BW achieves a higher joint likelihood than unconstrained BW.
That improvement shows the benefit of constraint information **on the joint
objective**; it does not resolve the mismatch with the constrained generative law.

The generating parameters also score below the joint BW fit on this objective. The
parameter comparison below illustrates why a better joint score should not be
read as parameter recovery. EM's ascent property concerns the objective being
optimized, even when the model is misspecified; it guarantees neither a global
optimum nor this particular ranking of fits.

In [ ]:
print("Fitted vs true emissions — deliberately not close:\n")
# Columns follow each model's observed_to_external, which load_model sorts --
# it is not OBSERVED_STATES.
def emission_frame(model, prefix):
    return pd.DataFrame(
        np.round(np.asarray(model.emission_mat, dtype=float), 3),
        index=model.hidden_to_external,
        columns=model.observed_to_external,
    ).add_prefix(prefix)


print(pd.concat([
    emission_frame(true_hmm, "true: "),
    emission_frame(fitted, "fit: "),
], axis=1).to_string())

print("\nstart vector")
print(f"  true {np.round(np.asarray(true_hmm.start_vec, dtype=float), 3)}")
print(f"  fit  {np.round(np.asarray(fitted.start_vec, dtype=float), 3)}")

## 5. Generalized EM for the constrained distribution

Let $C_T$ be the event that the hidden path satisfies every constraint over
horizon $T$, and $Z_T(\theta)=P(C_T\mid\theta)$. For observations drawn from the
constrained distribution, maximize

$$\ell_{\mathrm{conditional}}(\theta)
=\sum_i\left[\log P(y_i,C_{T_i}\mid\theta)-\log Z_{T_i}(\theta)\right].$$

The E-step is the same as joint BW: compute
$q_i(x)=P(x\mid y_i,C_{T_i},\theta_{\mathrm{old}})$ using constrained
forward-backward. The M-step increases the surrogate

$$Q(\theta\mid\theta_{\mathrm{old}})
=\sum_i E_{q_i}[\log P(x,y_i\mid\theta)]-\sum_i\log Z_{T_i}(\theta).$$

GEM holds the E-step posterior fixed while updating the chain, recomputes the
constraint-only statistics at each inner step, and backtracks until the surrogate
increases. It needs an improving M-step, not an exact maximizer. `inner_max_iter`
limits these steps; `step_size` and `max_backtracks` control the search, and
`inner_tol` controls inner stopping. Both learners preserve initial structural
zeros. GEM uses no pseudocounts and retains emission rows with no observed counts.

In [ ]:
from conin.hidden_markov_model.learning.generalized_em_mvr import generalized_em_mvr_chmm

comparison_iterations = 60

gem_fit, gem_history = generalized_em_mvr_chmm(
    start_model, sequences,
    max_iter=comparison_iterations, tol=0,
    inner_max_iter=10, inner_tol=1e-6,
    step_size=1.0, max_backtracks=30,
)
print(f"GEM conditional log-likelihood: {gem_history[0]:.3f} -> {gem_history[-1]:.3f}")
print(f"Smallest recorded increase: {np.diff(gem_history).min():.3e}")

Full sequences and partial `{time: label}` dictionaries can appear in the same
batch. Dictionaries require `time_horizons`: a shared integer or one horizon per
sequence. Unobserved times still drive transitions and constraints; only emission
counts omit them. The constraint normalizer covers the complete horizon, including
unobserved trailing times. Here we retain every other observation from eight runs.

In [ ]:
partial_sequences = [{t: seq[t] for t in range(0, T, 2)} for seq in sequences[:8]]
partial_fit, partial_history = generalized_em_mvr_chmm(
    start_model, partial_sequences, time_horizons=T,
    max_iter=3, tol=0,
)
print(f"Partial-data conditional log-likelihood: {partial_history[0]:.3f} -> {partial_history[-1]:.3f}")

## 6. Joint Baum–Welch versus conditional GEM

| | Joint Baum–Welch | Conditional GEM |
|---|---|---|
| Data interpretation | Feasibility is additional evidence about an HMM draw | Draws come from the distribution restricted to feasible paths |
| Objective | $\sum_i\log P(y_i,C_{T_i}\mid\theta)$ | $\sum_i\log P(y_i\mid C_{T_i},\theta)$ |
| E-step | Posterior given observations and feasibility | Same posterior |
| M-step | Normalize expected counts | Normalize emission counts; improve chain logits with the normalizer correction |

The rejection sampler in section 2 draws from the **conditional** distribution.
For this fixed-horizon batch,

$$\ell_{\mathrm{joint}}(\theta)
=\ell_{\mathrm{conditional}}(\theta)+N\log Z_T(\theta).$$

Joint BW therefore also rewards making feasible paths more probable under the
underlying HMM. GEM accounts for selection by conditioning on feasibility. Both
objectives are useful, but they answer different modeling questions.

Use the same data, initialization, and outer iteration budget for both fits.
Set BW's `pseudocount=0` to compare unregularized objectives. An outer GEM iteration
can contain several gradient steps and additional constraint passes, so equal
iteration counts do not imply equal computation time. Score the returned models
explicitly: on budget exhaustion each is one update ahead of its last history
entry. Raw histories optimize different quantities and should not be ranked
against each other.

In [ ]:
joint_fit, joint_history = baum_welch_mvr_chmm(
    start_model, sequences,
    max_iter=comparison_iterations, tol=0, pseudocount=0,
)


def objective_scores(hmm):
    """Score both objectives and feasibility at the common horizon T."""
    chmm = MVR_CHMM(hidden_markov_model=hmm, constraints=[no_A_early])
    joint = sum(forward_backward_mvr_chmm(chmm, seq)[2] for seq in sequences)
    log_z = forward_backward_mvr_chmm(chmm, {}, time_horizon=T)[2]
    return joint, joint - len(sequences) * log_z, math.exp(log_z)


comparison = pd.DataFrame(
    [objective_scores(hmm) for hmm in (start_model.hidden_markov_model, joint_fit, gem_fit, true_hmm)],
    index=["Initialization", "Joint BW", "Conditional GEM", "Generating model"],
    columns=["Joint log-likelihood", "Conditional log-likelihood", "P(C_T)"],
)
print(comparison.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name, trace, column, color in zip(
    axes,
    ["Joint BW", "Conditional GEM"],
    [joint_history, gem_history],
    ["Joint log-likelihood", "Conditional log-likelihood"],
    ["#2E6DB4", "#E08A3C"],
):
    scores = [*trace, comparison.loc[name, column]]
    ax.plot(range(len(scores)), scores, color=color, label=name)
    ax.axhline(comparison.loc["Generating model", column], color="0.4", linestyle="--", label="Generating model")
    ax.set(xlabel="Completed outer updates", ylabel=column, title=name)
    ax.grid(color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend()
fig.tight_layout()
plt.show()

Compare models **within each score column**. Increasing joint likelihood need not
increase conditional likelihood, or vice versa. The feasibility column shows how
much probability each underlying HMM assigns to the selection event. In this
seeded run, BW makes feasibility nearly certain, while GEM keeps it rare, as in
the generating model. BW scores higher on the joint objective; GEM scores higher
on the conditional objective.

Each learner improves its own objective; neither guarantees the best score among
all fits or recovery of the generating parameters. The conditional objective
matches this sampler, but a higher training score alone does not establish better
prediction on new data.

## Notes

- Joint BW maximizes $\log P(y, C_T \mid \theta)$; GEM maximizes
  $\log P(y \mid C_T, \theta)$. Both use the same constrained posterior,
  but GEM includes the parameter-dependent constraint normalizer in its M-step.
- `history[i]` is the log-likelihood at the **start** of iteration `i`, before
  that iteration's update, so `history[0]` scores the model passed in. If EM
  converged, `history[-1]` scores the returned model exactly; if it stopped at
  `max_iter` instead, one further update was applied and the returned model is a
  step ahead of `history[-1]`. That case warns, unless `tol <= 0`, which is an
  explicit request for exactly `max_iter` iterations.
- The fit runs on a `deepcopy`, so the model passed in is untouched and its
  constraint alignment survives.
- Parameters are written back in place rather than through `load_model`, which
  would re-derive the state ordering from sorted label names and could silently
  permute the internal indices.
- Emission counts accumulate only at observed times. An unobserved time still
  drives the chain and every MVR active at it, but contributes no emission
  statistic.
- `dtype` defaults to `float64` here, unlike Viterbi's `float32`: this recursion
  carries constraint-satisfaction probabilities whose spread grows with the
  horizon.
- With fixed constraints, a hidden-state permutation is equivalent only when
  it also preserves the constraint semantics.
- EM is a local method on a multimodal likelihood, and maximum likelihood on a
  finite sample does not return the generating parameters. Section 4 shows the
  true model scoring *worse* than the fit on the very objective being maximized.
  Judge a fit by its objective, not by its distance from parameters you happen to
  know.